<a href="https://colab.research.google.com/github/VAdarshReddy28/MLA0204-FOML/blob/main/Practical%20Problem%202%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

filename = list(uploaded.keys())[0]
data = pd.read_csv(filename)

print("Dataset:")
display(data)

Saving enjoysport.csv to enjoysport (1).csv
Dataset:


,sky,airtemp,humidity,wind,water,forcast,enjoysport
0,sunny,warm,normal,strong,warm,same,yes
1,sunny,warm,high,strong,warm,same,yes
2,rainy,cold,high,strong,warm,change,no
3,sunny,warm,high,strong,cool,change,yes


In [ ]:
import pandas as pd

# Load dataset
data = pd.read_csv(filename)

attributes = list(data.columns[:-1])
target = data.columns[-1]

# Check whether a hypothesis is more general or equal to another
def more_general_or_equal(h1, h2):
    for x, y in zip(h1, h2):
        if x != '?' and x != y:
            return False
    return True

# Generalize S
def generalize_S(s, example):
    new_s = s.copy()
    for i in range(len(s)):
        if s[i] == '0':
            new_s[i] = example[i]
        elif s[i] != example[i]:
            new_s[i] = '?'
    return new_s

# Specialize G
def specialize_G(g, example, domains):
    specializations = []
    for i in range(len(g)):
        if g[i] == '?':
            for value in domains[i]:
                if value != example[i]:
                    new_hypothesis = g.copy()
                    new_hypothesis[i] = value
                    specializations.append(new_hypothesis)
    return specializations

def candidate_elimination(data):
    # Get attribute values
    domains = []
    for attribute in attributes:
        domains.append(list(data[attribute].unique()))

    # Most specific hypothesis
    S = ['0'] * len(attributes)

    # Most general hypothesis
    G = [['?'] * len(attributes)]

    print("Initial S:", S)
    print("Initial G:", G)

    for index, row in data.iterrows():
        example = list(row[attributes])
        print(f"\n--------------------------------")
        print(f"Example {index + 1}")
        print("Attributes:", example)
        print("Class:", row[target])

        # Positive example
        if row[target].lower() == 'yes':
            # Remove from G any hypothesis inconsistent with the positive example
            G = [g for g in G if more_general_or_equal(g, example)]
            # Generalize S to cover the example
            S = generalize_S(S, example)
            # Remove from G any hypothesis that is not more general than S
            G = [g for g in G if more_general_or_equal(g, S)]

        # Negative example
        else:
            # Remove from S any hypothesis that covers the negative example (not applicable as S must only cover +ve)
            new_G = []
            for g in G:
                # If g covers the negative example, specialize it
                if more_general_or_equal(g, example):
                    specializations = specialize_G(g, example, domains)
                    for h in specializations:
                        # h must be consistent with S
                        if more_general_or_equal(h, S):
                            new_G.append(h)
                else:
                    new_G.append(g)

            # Filter out hypotheses that are more specific than other hypotheses in G
            G = []
            for g in new_G:
                if not any(more_general_or_equal(g_prime, g) and g_prime != g for g_prime in new_G):
                    if g not in G:
                        G.append(g)

        print("S:", S)
        print("G:", G)

    return S, G

S, G = candidate_elimination(data)

print("\n======================================")
print("FINAL CANDIDATE-ELIMINATION RESULT")
print("======================================")

print("\nSpecific Boundary (S):")
print(S)

print("\nGeneral Boundary (G):")
for g in G:
    print(g)

Initial S: ['0', '0', '0', '0', '0', '0']
Initial G: [['?', '?', '?', '?', '?', '?']]

--------------------------------
Example 1
Attributes: ['sunny', 'warm', 'normal', 'strong', 'warm', 'same']
Class: yes
S: ['sunny', 'warm', 'normal', 'strong', 'warm', 'same']
G: [['?', '?', '?', '?', '?', '?']]

--------------------------------
Example 2
Attributes: ['sunny', 'warm', 'high', 'strong', 'warm', 'same']
Class: yes
S: ['sunny', 'warm', '?', 'strong', 'warm', 'same']
G: [['?', '?', '?', '?', '?', '?']]

--------------------------------
Example 3
Attributes: ['rainy', 'cold', 'high', 'strong', 'warm', 'change']
Class: no
S: ['sunny', 'warm', '?', 'strong', 'warm', 'same']
G: [['sunny', '?', '?', '?', '?', '?'], ['?', 'warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', 'same']]

--------------------------------
Example 4
Attributes: ['sunny', 'warm', 'high', 'strong', 'cool', 'change']
Class: yes
S: ['sunny', 'warm', '?', 'strong', '?', '?']
G: [['sunny', '?', '?', '?', '?', '?'], ['?'